# Regression
---
## Marketing tool
L’obiettivo di questo notebook è sviluppare un modello che, analizzando i rating sia in grado di predirre il prezzo del telefono.
Questo sistema può essere utilizzato come strumento di marketing per personalizzare le offerte, migliorare l’esperienza utente e supportare decisioni strategiche basate sui dati.

### Import dataset


In [6]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
from tensorflow import keras
import tensorflow as tf
import seaborn as sns
import pandas as pd
import numpy as np

df = pd.read_csv("./data/Mobile Reviews Sentiment.csv")

### Pulizia dataset

In [7]:
features = ["battery_life_rating", "camera_rating", "performance_rating", "design_rating", "display_rating"]
colonne_da_tenere = features + ["price_usd"]
df_clean = df[colonne_da_tenere].copy()

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   battery_life_rating  50000 non-null  int64  
 1   camera_rating        50000 non-null  int64  
 2   performance_rating   50000 non-null  int64  
 3   design_rating        50000 non-null  int64  
 4   display_rating       50000 non-null  int64  
 5   price_usd            50000 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 2.3 MB


### Scaling

In [8]:
X = df_clean[features]
y = df_clean['price_usd']

X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(40000, 5) (10000, 5)
(40000,) (10000,)


### Rete neurale

In [9]:
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="relu", input_shape=X_train.shape[1:]),
    keras.layers.Dense(15, activation="relu"),
    keras.layers.Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

C:\Users\julic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 30)             │           180 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 15)             │           465 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            16 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 661 (2.58 KB)

 Trainable params: 661 (2.58 KB)

 Non-trainable params: 0 (0.00 B)

### Addrestramento

In [11]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=10,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-5
    )
]
history = model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test), callbacks=callbacks)

Epoch 1/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 695us/step - loss: 96355.8359 - mae: 254.9135 - val_loss: 96532.2656 - val_mae: 256.2786 - learning_rate: 0.0010
Epoch 2/20
 298/1250 ━━━━━━━━━━━━━━━━━━━━ 0s 508us/step - loss: 98156.7706 - mae: 256.5915

C:\Users\julic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_auc` which is not available. Available metrics are: loss,mae,val_loss,val_mae
  current = self.get_monitor_value(logs)
C:\Users\julic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\callbacks\callback_list.py:171: UserWarning: Learning rate reduction is conditioned on metric `val_auc` which is not available. Available metrics are: loss,mae,val_loss,val_mae,learning_rate.
  callback.on_epoch_end(epoch, logs)


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 654us/step - loss: 96337.9141 - mae: 254.8098 - val_loss: 96960.7578 - val_mae: 258.4176 - learning_rate: 0.0010
Epoch 3/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 667us/step - loss: 96355.0234 - mae: 254.9792 - val_loss: 96584.1562 - val_mae: 256.5893 - learning_rate: 0.0010
Epoch 4/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 645us/step - loss: 96357.2344 - mae: 254.8948 - val_loss: 96540.2656 - val_mae: 256.2860 - learning_rate: 0.0010
Epoch 5/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 657us/step - loss: 96346.1719 - mae: 254.8331 - val_loss: 96593.3359 - val_mae: 256.8459 - learning_rate: 0.0010
Epoch 6/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 647us/step - loss: 96331.1328 - mae: 254.9104 - val_loss: 96472.2188 - val_mae: 255.9457 - learning_rate: 0.0010
Epoch 7/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 646us/step - loss: 96342.9688 - mae: 254.8843 - val_loss: 96591.4688 - val_mae: 254.9743 - learning_rate: 0.0010
Epoch 8/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 650us/step - loss: